# Solutions · Chapter 01-01 · Python and data diagnostic

The answers, plus why each one is the idiom you will see everywhere else. Several tasks have a
version that works and a version that generalises - both are shown, because knowing the
difference is most of what "fluent" means.

Every check in this notebook passes, so you can see the expected shape of each result.

In [ ]:
import numpy as np
import pandas as pd

score = {}

def check(task, answer, expected, note=""):
    if answer is None:
        score[task] = None
        print(f"task {task}: not attempted")
        return
    try:
        if isinstance(expected, (pd.DataFrame, pd.Series)):
            ok = expected.equals(answer)
        elif isinstance(expected, np.ndarray):
            ok = np.allclose(np.asarray(answer), expected)
        elif isinstance(expected, float):
            ok = abs(float(answer) - expected) < 1e-6
        else:
            ok = answer == expected
    except Exception as exc:
        ok, note = False, f"{note} ({type(exc).__name__})"
    score[task] = bool(ok)
    print(f"task {task}: {'PASS' if ok else 'FAIL'}{'  - ' + note if note and not ok else ''}")

## Task 1 · `average`

In [ ]:
def average(values):
    if not values:                 # empty list, empty tuple, None-ish - all falsy
        return None
    return sum(values) / len(values)

check(1, average([2, 4, 9]), 5.0)
print("empty list ->", average([]))

`sum(values) / len(values)` is the whole calculation. The interesting part is the guard.

**Why `if not values` and not `if len(values) == 0`:** both work, and the first is the Python
idiom - it reads as "if there is nothing here". It also handles anything empty, not just lists.

**Why return `None` rather than 0:** an average of no numbers does not exist, and 0 is a specific
claim that would flow silently into whatever comes next. This distinction matters more than it
looks: chapter 02-04 is largely about the damage done by filling an absent value with a
plausible-looking number.

**The common wrong answer** is no guard at all, which raises `ZeroDivisionError` on an empty
list. In a notebook that is obvious. In a `groupby` over a column where one category happens to
be empty, it stops a pipeline at 3am.

## Task 2 · Filtering records

In [ ]:
readings = [
    {"name": "north", "temp": 18.5}, {"name": "south", "temp": 24.1},
    {"name": "east", "temp": 20.0}, {"name": "west", "temp": 27.3},
]

warm_stations = [r["name"] for r in readings if r["temp"] > 20]

check(2, warm_stations, ["south", "west"])

A list comprehension: *the thing I want*, *where it comes from*, *which ones*. Read it left to
right as "the name of each reading, for every reading, where the temperature is above 20".

**The trap in this task was `east` at exactly 20.0.** "Above 20" excludes it; `>=` would include
it. Boundary conditions are where requirements are ambiguous and where bugs live - if a
specification says "above", check whether the person meant it, because half the time they did not.

**The longer version** is fine too:

```python
warm_stations = []
for r in readings:
    if r["temp"] > 20:
        warm_stations.append(r["name"])
```

Six lines instead of one, and identical behaviour. Comprehensions are worth the fluency because
you will read hundreds of them, but there is no prize for compressing a complicated loop into an
unreadable one - if a comprehension needs two conditions and a nested loop, write the loop.

## Task 3 · Counting

In [ ]:
words = ["rain", "sun", "rain", "fog", "sun", "rain"]

counts = {}
for word in words:
    counts[word] = counts.get(word, 0) + 1

check(3, counts, {"rain": 3, "sun": 2, "fog": 1})

`counts.get(word, 0)` returns 0 when the key is absent, so the first sighting of a word is
handled by the same line as the hundredth. Without it you need an `if word in counts` branch.

**In real work you would use `collections.Counter(words)`**, which does this in one call. The
task banned it because the pattern "look up with a default, then update" appears constantly in ML
code - mapping category names to codes, accumulating errors per segment, tallying predictions -
and it is worth having in your fingers.

**In pandas** the same job is `pd.Series(words).value_counts()`, which you will use from module 02
onwards. Three tools, one idea; knowing that they are one idea is the point.

## Task 4 · Boolean masks

In [ ]:
values = np.arange(10)
mask = values > 5
big_values = values[mask]

print("the mask itself:", mask)
check(4, big_values, np.array([6, 7, 8, 9]))

Written out in two steps so the middle object is visible, because **the mask is the concept**.

`values > 5` does not return `True`. It returns an array of ten booleans, one per element.
Indexing with that array keeps the positions where it is `True`. Every filter you write for the
rest of this course - in NumPy, in pandas, on rows or columns - is this same two-step move.

Normally you write it as one line, `values[values > 5]`, and now you know what the inner part
evaluates to.

**Why this matters later:** in pandas, `df[df["units"] >= 10]` is the identical mechanism, and
so is `errors[errors > threshold]` when you are hunting the worst predictions in 07-05. Learning
it as "the filter syntax" makes each of those a separate fact; learning it as "a boolean array
selects positions" makes them one.

## Task 5 · Reshape and axes

In [ ]:
flat = np.arange(12) * 2.0
grid = flat.reshape(3, 4)
column_means = grid.mean(axis=0)         # axis=0 collapses the rows, leaving one value per column

print(grid)
print("shape", grid.shape, "-> column means shape", column_means.shape)
check(5, column_means, np.array([8.0, 10.0, 12.0, 14.0]))

**The rule worth memorising:** `axis=n` is the axis that **disappears**.

`grid` has shape `(3, 4)`. Averaging with `axis=0` removes the 3 and leaves shape `(4,)` - four
column means. Averaging with `axis=1` removes the 4 and leaves shape `(3,)` - three row means.

People try to remember "axis 0 is columns" and get it backwards under pressure, because axis 0
*is* the row axis and it is the one being collapsed. "The axis you name is the one that
disappears" survives contact with 3-D arrays too, which the other phrasing does not.

**Why this is the most dangerous task on the diagnostic:** both versions run. Both return an
array of plausible numbers. Nothing raises. In a scaling step, taking means over the wrong axis
silently standardises each *sample* instead of each *feature*, and the model still trains, still
produces a score, and is quietly wrong. Printing `.shape` costs one line and catches it
immediately - which is why nearly every code cell in module 10 prints a shape.

## Task 6 · Broadcasting

In [ ]:
M = np.array([[1.0, 10.0], [3.0, 20.0], [5.0, 30.0]])

centred = M - M.mean(axis=0)

print("M.shape", M.shape, " column means shape", M.mean(axis=0).shape)
print("result column means:", centred.mean(axis=0))
check(6, centred, np.array([[-2.0, -10.0], [0.0, 0.0], [2.0, 10.0]]))

`M` has shape `(3, 2)`, `M.mean(axis=0)` has shape `(2,)`, and NumPy stretches the smaller one
across the rows so the subtraction happens column by column. That stretching is **broadcasting**,
and it is why numerical Python code has so few loops.

The rule: shapes are lined up from the right, and a dimension of size 1 (or absent) is repeated.
`(3, 2)` against `(2,)` works. `(3, 2)` against `(3,)` does not, and raises - which is the good
case, because it fails loudly.

The bad case is when it works and you did not mean it. `(3, 1)` against `(1, 3)` broadcasts to
`(3, 3)`, turning two small arrays into a bigger one with no error at all. If an array is
mysteriously the wrong size, look for that.

**This exact operation is feature centring**, and with a division by the standard deviation it is
standardisation, which chapter 04-06 does properly - including the part this cell quietly gets
wrong for real work: the mean must come from the *training* rows only, not from all of `M`.

## Task 7 · Selecting rows and columns

In [ ]:
sales = pd.DataFrame({
    "shop": ["a", "b", "c", "a", "b"],
    "units": [12, 7, 30, 9, 15],
    "price": [2.5, 3.0, 1.5, 2.5, 3.0],
})

busy = sales.loc[sales["units"] >= 10, ["shop", "units"]]

print(busy)
check(7, busy, sales.loc[sales["units"] >= 10, ["shop", "units"]])

`.loc[rows, columns]` does both selections in one step: a boolean mask for the rows, a list of
names for the columns.

**The version to unlearn:** `sales[sales["units"] >= 10][["shop", "units"]]`. It gives the same
answer here, and it is two operations - select, then select again from the result. That is
**chained indexing**, and when you assign to it (`df[mask]["col"] = value`) pandas may modify a
temporary copy and silently discard your change. `.loc` in one call always acts on the original.

Notice the index of the result: `0, 2, 4`. The original row labels are kept, not renumbered. That
is usually what you want - it means you can still trace a row back - but it surprises people who
expect `0, 1, 2`. If you need fresh numbering, `.reset_index(drop=True)` is explicit about it.

## Task 8 · groupby

In [ ]:
units_per_shop = sales.groupby("shop")["units"].sum()

print(units_per_shop)
check(8, units_per_shop, sales.groupby("shop")["units"].sum())

Split by `shop`, take the `units` column, add it up. The result is a Series indexed by shop, and
it is sorted by the group key by default - which is why the task could ask for it sorted without
any extra work.

**`groupby` is the single most valuable pandas verb in this course.** Not because summing units is
hard, but because "compute this metric separately for each group" *is* error analysis. Every time
a later chapter asks where a model fails - by weather, by hour, by customer segment, by class -
the answer is a `groupby` on the errors. You met one in 00-01 and will meet dozens more.

Two variants worth knowing now:

```python
sales.groupby("shop")["units"].agg(["count", "mean", "max"])   # several statistics at once
sales.groupby(["shop", "price"])["units"].sum()                # group by more than one column
```

## Task 9 · Joining, and counting the rows

In [ ]:
orders = pd.DataFrame({"order_id": [1, 2, 3, 4], "customer_id": [10, 11, 10, 99]})
customers = pd.DataFrame({"customer_id": [10, 11, 12], "city": ["Bonn", "Kiel", "Jena"]})

joined = orders.merge(customers, on="customer_id", how="left")

print(joined)
print("\nrows in, rows out:", len(orders), "->", len(joined))
check(9, joined, orders.merge(customers, on="customer_id", how="left"))

**Four rows in, four rows out.** `how="left"` keeps every order; order 4's customer (99) does not
exist, so its `city` is `NaN`. Customer 12 has no orders and simply does not appear.

The four kinds of join, and what each is *for*:

| `how=` | Keeps | Use when |
|---|---|---|
| `"left"` | every row of the left frame | You are enriching a fact table - the default for adding attributes to observations |
| `"inner"` | only rows matching on both sides | You genuinely want to drop unmatched rows - and you should say so out loud, because it silently deletes data |
| `"outer"` | everything from both | Reconciling two sources, checking what is missing where |
| `"right"` | every row of the right frame | Rare; usually a `left` join written backwards |

**Why the row count was the real question.** If `customers` had contained `customer_id` 10 twice
- a duplicate, a second address, a historical record - then orders 1 and 3 would each match two
rows and the result would have **six** rows, not four. Nothing warns you. Your dataset now has
duplicated observations, they end up on both sides of a train/test split, the model recognises
rows it has already seen, and the score is inflated. That is duplicate leakage, and 04-05 shows
it happening.

**The habit:** print `len()` before and after every join, every time. If the number changed and
you did not expect it to, stop and find out why. One line, and it catches a whole family of
silent disasters.

## Task 10 · Timestamps

In [ ]:
log = pd.DataFrame({
    "when": ["2024-03-01 09:00", "2024-03-01 17:30", "2024-03-02 08:15", "2024-03-02 19:45"],
    "amount": [5, 7, 2, 6],
})

log["when"] = pd.to_datetime(log["when"])
daily_total = log.groupby(pd.Grouper(key="when", freq="D"))["amount"].sum()

print(log.dtypes.to_string())
print()
print(daily_total)
check(10, daily_total, log.groupby(pd.Grouper(key="when", freq="D"))["amount"].sum())

Two steps, and the first is the one people skip.

**`pd.to_datetime` converts text to real timestamps.** Before it, `when` is a string column:
sorting it works only because ISO format happens to sort alphabetically, and "the day part" would
have to be extracted with string slicing. After it, the column knows it is time, and
`.dt.hour`, `.dt.dayofweek`, comparison with a date, and grouping by period all become available.
Check `.dtypes` after loading any file - a date column read as text is one of the most common
quiet defects in a dataset, and module 02 says so at more length.

**Grouping by day.** `pd.Grouper(key="when", freq="D")` buckets timestamps into calendar days.
Alternatives you will see: `log.set_index("when").resample("D")["amount"].sum()` does the same
thing, and `log.groupby(log["when"].dt.date)` also works but gives you a plain-date index rather
than a proper time index, which matters when you later want to plot it or fill in missing days.

**The one that will bite you later:** `resample` and `Grouper` produce rows for periods with **no
data**, as zeros or NaN, whereas grouping by `.dt.date` silently omits them. When a day with no
rentals disappears from a time series, every lag feature after it is shifted by one and every
number downstream is wrong. Module 09 opens with this.

---

## Where to go next

Whatever you scored, the routing advice in the chapter stands: 9 or more, skip to **02-01**;
6 to 8, read only the chapters your failures point to; fewer than 6, work through **01-02** to
**01-06** in order.

One thing worth saying if you scored badly: the ten tasks above are a *skill* diagnostic, not an
aptitude one. Nothing here is difficult, and all of it is muscle memory that comes from typing it
a few dozen times. Five hours of module 01 buys you the rest of the course without a syntax fight
in the way.